# WIKI LM

项目目标：用维基百科语料训练一个百科型LLM，然后通过微调让他成为对话型百科全书。

项目流程：
1. 准备数据库（wikitext-2/wikitext-103）并使用tiktoken分词
2. 搭建LM网络
3. 编写训练算法
4. 训练
5. 微调

## Phase 1: 准备数据集

轻量级：Wikitext-2（适用于原理验证、快速迭代）

重量级：Wikitext-103（用于正式训练）

无论最终语言模型要执行什么特定领域的任务，最基础的自然语言能力都是不可或缺的。根据学界经验，仅仅依靠特定领域的高质量数据集是远远不够的，必须要在训练集中混合更广泛的语料库数据，才能为模型提供充足的语言知识。

### dataset库的基础用法

- load_dataset      从 Hugging Face 或本地加载数据

- print 查看结构     看 split、字段、样本

- select/filter     抽样、筛选

- map               批量处理数据

- train_test_split  划分数据集

- save_to_disk      保存成本地 datasets 格式

- load_from_disk    从本地重新读取

In [1]:
from pathlib import Path
from datasets import load_dataset, load_from_disk

local_path = Path("wikitext-103-raw-v1")

# 下载百科文章数据集 wikitext-103-raw-v1，或者从本地加载（如果之前已经下载过）
if local_path.exists():
    print("从本地读取数据集")
    ds = load_from_disk(str(local_path))
else:
    print("本地不存在，开始从 Hugging Face 下载")
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
    ds.save_to_disk(str(local_path))

print(ds)
print(ds["train"][1])
print(ds["train"][3])



/home/isi/weiyu/.pyenv/versions/anaconda3-5.3.1/envs/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


从本地读取数据集
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
{'text': ' = Valkyria Chronicles III = \n'}
{'text': ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperi

通过 print(ds) 我们可以看到，dataset 的数据库是一个包含 'train', 'validation', 'test' 三个子集（split）的 DatasetDict，通过 print 可以查看数据集的各种基本信息。

```python
ds                     # 整个数据集字典
ds["train"]            # 训练集的一张表（子集）
ds["train"].features   # 子集的表头结构
ds["train"][0]         # 训练集第一行（example）
ds["train"][0]["text"] # 第一行里的文本
```

---

### WikiText数据集的预处理

WikiText 的原始结构是这样的：

```python
{"text": ""}
{"text": " = Valkyria Chronicles III = \n"}
{"text": ""}
{"text": " Senjō no Valkyria 3 : Unrecorded Chronicles ..."}
{"text": ""}
{"text": " = = Gameplay = = \n"}
```

WikiText 把文章按行保存为 example，而非按照段落保存。文章各个层级的标题以
```python
= Article Title =
= = Section Title = =
= = = Subsection Title = = =
```
的形式被保存为单独的 example，而段落之间的空行（段落间距）同样以空 example 的形式被保存了下来。
这样细碎的数据形式不能很好地满足 pre-train 对连贯文本的需求，因此必须对先数据库进行清洗和拼接，使其成为如下的连续语料：
```python
= Article Title =

Paragraph 1.

= = Section = =

Paragraph 2.
```

In [2]:
from pathlib import Path

out_dir = Path("data/wikitext103_clean_txt")


def normalize_line(text: str) -> str:
    """
    对单行 WikiText 做轻量清洗。
    注意：不删除 = Title = 这类标题结构。
    """
    text = text.rstrip("\n")
    text = text.rstrip()
    return text


def save_split_to_txt(split_name: str):
    out_file = out_dir / f"{split_name}.txt"

    with out_file.open("w", encoding="utf-8") as f:
        previous_blank = False

        for ex in ds[split_name]:
            line = normalize_line(ex["text"])

            if line.strip() == "":
                if not previous_blank:
                    f.write("\n")
                    previous_blank = True
                continue

            f.write(line + "\n")
            previous_blank = False

    print(f"saved: {out_file}")


if out_dir.exists():
    print(f"目录 {out_dir} 已存在，跳过保存")
else:
    out_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train", "validation", "test"]:
        save_split_to_txt(split)

目录 data/wikitext103_clean_txt 已存在，跳过保存


得到 clean text 后，在全体（包含 train、test、validation 三个 split）文本上训练专属的 BPE。

- 问: 为什么要训练 BPE 针对于训练集的？

  - 答: 可以更好地适应语料特有的格式，特别是各种符号的特殊用法，如 wikitext 的标题格式 "= =" 以及 特殊符号预处理标记 @：

All three regiments of the NK 2nd Division @-@ the 4th , 17th , and 6th , in line from north to south @-@ crossed during the night to the east side of the Naktong River into the 23rd Regiment sector .

In [3]:
from tokenizers import ByteLevelBPETokenizer
from pathlib import Path

paths = [
    "data/wikitext103_clean_txt/train.txt"
]

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=paths,
    vocab_size=32000,
    min_frequency=2,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ],
)

out_dir = Path("tokenizer-wikitext103-bpe")
out_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save_model(str(out_dir))

print("tokenizer saved")




tokenizer saved


在完成分词工作后，将 txt 的连续文本切成固定长度的　token blocks。

用固定长度文本切片进行训练能提高效率，但同时也意味着模型只能学到 block 内的上下文。

```python
加载文本
    text_ds = train.txt
tokenizer对文本编码
    ids = tokenizer.encode(text_ds)
切片 + 截断多余文本（可选：overlap）
    block_size = 512
    num_block = len(text_ds) // block_size
    ids = ids[:num_block * block_size]
    i_block = []
    for i in range(0, len(text_ds), block_size):
        i_block.append(text_ds[i:i+block_size])
    label = [x.copy() for x in i_block]
保存切片到硬盘
    save_to_disk(i_block, label)
```

In [4]:
from pathlib import Path
from datasets import Dataset, DatasetDict
from tokenizers import ByteLevelBPETokenizer

block_size = 512

files = {
    "train": "data/wikitext103_clean_txt/train.txt",
    "validation": "data/wikitext103_clean_txt/validation.txt",
    "test": "data/wikitext103_clean_txt/test.txt",
}

tokenizer = ByteLevelBPETokenizer(
    "tokenizer-wikitext103-bpe/vocab.json",
    "tokenizer-wikitext103-bpe/merges.txt",
)


def file_to_blocks(path):
    text = Path(path).read_text(encoding="utf-8")

    ids = tokenizer.encode(text).ids

    # 因为 labels 需要向后取 1 个 token，
    # 所以至少要预留 len(ids) - 1 的长度。
    total_length = len(ids) - 1

    # 截断到 block_size 的整数倍
    total_length = (total_length // block_size) * block_size

    input_ids = [
        ids[i:i + block_size]
        for i in range(0, total_length, block_size)
    ]

    labels = [
        ids[i + 1:i + block_size + 1]
        for i in range(0, total_length, block_size)
    ]

    return Dataset.from_dict({
        "input_ids": input_ids,
        "labels": labels,
    })


save_path = Path("data/wikitext103_bpe_lm_block512")

if save_path.exists():
    print(f"切片数据已存在，直接加载：{save_path}")
    lm_ds = load_from_disk(str(save_path))
else:
    print("切片数据不存在，开始生成...")

    lm_ds = DatasetDict({
        split: file_to_blocks(path)
        for split, path in files.items()
    })

    lm_ds.save_to_disk(str(save_path))
    print(f"切片数据已保存：{save_path}")

print(lm_ds)
print(lm_ds["train"][0].keys())
print(len(lm_ds["train"][0]["input_ids"]))

切片数据已存在，直接加载：data/wikitext103_bpe_lm_block512
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 223951
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 469
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 534
    })
})
dict_keys(['input_ids', 'labels'])
512


至此，数据的预处理告一段落。

---

## Phase 2: 预训练准备

把 DatasetDict 转成 PyTorch 可用格式，然后构建 DataLoader

In [5]:
from torch.utils.data import DataLoader

lm_ds.set_format(type="torch", columns=["input_ids", "labels"])

batch_size = 16

train_loader = DataLoader(
    lm_ds["train"],
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    lm_ds["validation"],
    batch_size=batch_size,
    shuffle=False  # 验证集不需要打乱
)

搭建语言模型: 

- Multi-Head Attention
- Feed Forward
- Layer Norm

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F


class Head(nn.Module):
    """ 一个单头自注意力层 """
    def __init__(self, embed_dim, head_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # 计算注意力分数 (Affinity)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # 融合 Value
        v = self.value(x)
        out = wei @ v # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """ 多个并行运行的自注意力头 """
    def __init__(self, embed_dim, num_heads, head_size):
        super().__init__()
        # 构建多头注意力模型
        # 提示: 使用 nn.ModuleList 存放多个独立的 Head
        self.heads = nn.ModuleList([Head(embed_dim, head_size) for _ in range(num_heads)])
        # 最后通过一个线性层投影回原始维度 (可选，但推荐)
        self.proj = nn.Linear(num_heads * head_size, embed_dim)

    def forward(self, x):
        # 拼接所有头的输出
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)

        return out


class FeedForward(nn.Module):
    """ 一个简单的线性层，后跟一个非线性激活函数 """
    def __init__(self, embed_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            # 实现第一层线性变换，将维度从 n_embed 扩大到 4 * n_embed
            # 添加 ReLU 激活函数
            # 实现第二层线性变换，将维度缩回到 n_embed
            # 添加一个投影层的 Dropout (概率设为 0.2)
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
    

class LayerNorm(nn.Module):
    """
    LayerNorm 的简化实现。注意它与 BatchNorm 的区别：
    BatchNorm 是在 Batch 维度归一化，而 LayerNorm 是在 Channel 维度归一化。
    """
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        # 用nn.Parameter初始化两个可学习参数：gamma (缩放) 和 beta (平移)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # 计算 x 在最后一个维度上的均值和方差
        # 执行归一化：(x - mean) / sqrt(var + eps)
        # 应用 gamma 和 beta 进行仿射变换
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)

        x = (x - mean) / torch.sqrt(var + self.eps)
        x = x * self.gamma + self.beta

        return x


class Block(nn.Module):
    """ Transformer Block: 通信 (Attention) + 计算 (FFN) """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding 维度, n_head: 我们想要的头数
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_embed, n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)

    def forward(self, x):
        # 实现 Pre-Norm 结构下的残差连接
        # 1. x = x + Attention(LayerNorm(x))
        # 2. x = x + FFN(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, block_size, n_embed):
        super().__init__()

        position = torch.arange(block_size).unsqueeze(1)  # (block_size, 1)

        div_term = torch.exp(
            torch.arange(0, n_embed, 2) * (-torch.log(torch.tensor(10000.0)) / n_embed)
        )  # (n_embed / 2,)

        pe = torch.zeros(block_size, n_embed)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 不是可学习参数，但会跟随模型保存和移动到 GPU
        self.register_buffer("pe", pe)

    def forward(self, T):
        return self.pe[:T]
    

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size):
        super().__init__()
        # 1. Token 嵌入表
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # 2. 位置编码 嵌入表 (让模型知道字符的顺序)
        self.position_encoding = SinusoidalPositionalEncoding(block_size, n_embed)
        # 3. 堆叠多个 Transformer Block
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        # 4. 最后的层归一化
        self.ln_f = LayerNorm(n_embed)
        # 5. 语言模型头 (从特征空间映射回词表大小)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # tok_emb: (B, T, C), pos_emb: (T, C)
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_encoding(T)
        x = tok_emb + pos_emb # 融合内容信息和位置信息

        x = self.blocks(x)    # 通过所有 Block
        x = self.ln_f(x)      # 最终归一化
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 限制上下文长度不能超过 block_size
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # 只关注最后一个时刻
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

---

## Phase 3: 正式训练

各组件构建完成，实例化语言模型。

In [ ]:
vocab_size = tokenizer.get_vocab_size()
print(f"vocab_size: {vocab_size}")

embed_dim = 512  # embedding dimension
n_head = 8
n_layer = 8  # transformer block 的数量
learning_rate = 3e-4
norm_clip = 1.0
max_iters = 10000

cuda_id = 0
device = torch.device("cuda:{}".format(cuda_id) if torch.cuda.is_available() else "cpu")

gpt_model = GPTLanguageModel(
    vocab_size=vocab_size,
    n_embed=embed_dim,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
).to(device)

# print(gpt_model)
total_params = sum(p.numel() for p in gpt_model.parameters())
print(f"模型参数数量: {total_params:,}")

vocab_size: 32000


选定优化器

In [8]:
opt = torch.optim.AdamW(gpt_model.parameters(), 
                        lr=learning_rate, 
                        weight_decay=1e-2,
                        betas=(0.90, 0.95),
                        eps=1e-8)

warmup_iters = 1000
min_lr = learning_rate * 0.1

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    opt,
    start_factor=1e-5,
    end_factor=1.0,
    total_iters=warmup_iters
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=max_iters - warmup_iters,
    eta_min=min_lr
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    opt,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_iters]
)

train_iter = iter(train_loader)
val_iter = iter(val_loader)

for i in range(max_iters):
    gpt_model.train()

    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    xb = batch["input_ids"].to(device)
    yb = batch["labels"].to(device)

    logits, loss = gpt_model(xb, yb)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt_model.parameters(), norm_clip)
    opt.step()
    scheduler.step()

    if i % 500 == 0:
        print(f"iter {i}: train loss {loss.item():.4f}")

        val_losses = []
        gpt_model.eval()

        with torch.no_grad():
            try:
                val_batch = next(val_iter)
            except StopIteration:
                val_iter = iter(val_loader)
                val_batch = next(val_iter)
            xb, yb = val_batch["input_ids"].to(device), val_batch["labels"].to(device)
            _, loss = gpt_model(xb, yb)
            val_losses.append(loss.item())
        avg_val_loss = sum(val_losses) / len(val_losses)

        print(f"iter {i}: validation loss {avg_val_loss:.4f}")

print("训练完成，开始评估...")
gpt_model.eval()

val_losses = []
with torch.no_grad():
    for batch in val_loader:
        xb, yb = batch["input_ids"], batch["labels"]
        xb, yb = xb.to(device), yb.to(device)
        _, loss = gpt_model(xb, yb)
        val_losses.append(loss.item())
avg_val_loss = sum(val_losses) / len(val_losses)
print(f"平均验证损失: {avg_val_loss:.4f}")

print("生成示例文本:")
context = "The history of natural language processing (NLP) started in the 1950s"
context_ids = torch.tensor(tokenizer.encode(context).ids, dtype=torch.long, device=device).unsqueeze(0)
generated_ids = gpt_model.generate(context_ids, max_new_tokens=100)[0].tolist()
generated_text = tokenizer.decode(generated_ids)
print(generated_text)


iter 0: train loss 10.5260
iter 0: validation loss 10.5144
iter 500: train loss 6.6352
iter 500: validation loss 6.4972
iter 1000: train loss 6.1487
iter 1000: validation loss 6.1278
iter 1500: train loss 5.8121
iter 1500: validation loss 5.5403
iter 2000: train loss 5.4794
iter 2000: validation loss 5.3177
iter 2500: train loss 5.1524
iter 2500: validation loss 5.1840
iter 3000: train loss 5.3038
iter 3000: validation loss 4.9507
iter 3500: train loss 5.2496
iter 3500: validation loss 4.8567
iter 4000: train loss 4.9541
iter 4000: validation loss 4.7206
iter 4500: train loss 4.7521
iter 4500: validation loss 4.9549
iter 5000: train loss 4.9330
iter 5000: validation loss 4.7767
iter 5500: train loss 4.7075
iter 5500: validation loss 4.6654
iter 6000: train loss 4.7829
iter 6000: validation loss 4.5665
iter 6500: train loss 4.8599
iter 6500: validation loss 4.1958
iter 7000: train loss 4.5636
iter 7000: validation loss 4.6611
iter 7500: train loss 4.5837
iter 7500: validation loss 4.678

In [13]:
import math

batch_size = 16

test_loader = DataLoader(
    lm_ds["test"],
    batch_size=batch_size,
    shuffle=False
)
test_iter = iter(test_loader)

@torch.no_grad()
def evaluate_metrics(model, data_iter, num_batches=50):
    model.eval()
    total_loss = 0
    total_tokens = 0

    for _ in range(num_batches):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(test_loader)
            batch = next(data_iter)

        xb = batch["input_ids"].to(device)
        yb = batch["labels"].to(device)

        _, loss = model(xb, yb)

        total_loss += loss.item() * batch_size * block_size
        total_tokens += batch_size * block_size

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    model.train()

    return {
        "Loss": avg_loss,
        "Perplexity": perplexity
    }


test_ids = []

for item in lm_ds["test"]:
    test_ids.extend(item["input_ids"])

metrics = evaluate_metrics(gpt_model, test_iter)

print("测试集评估结果:")
print(f"- Cross Entropy Loss: {metrics['Loss']:.4f}")
print(f"- Perplexity: {metrics['Perplexity']:.2f}")

测试集评估结果:
- Cross Entropy Loss: 3.5393
- Perplexity: 34.44


训练完成后保存 checkpoint，供 SFT 之用。

In [16]:
# =========================
# 保存模型 checkpoint
# =========================
import os

save_dir = "model/gpt_wikitext103_layer10"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "checkpoint.pt")

total_params = sum(p.numel() for p in gpt_model.parameters())

checkpoint = {
    "model_state_dict": gpt_model.state_dict(),
    "optimizer_state_dict": opt.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),

    "vocab_size": vocab_size,
    "embed_dim": embed_dim,
    "n_head": n_head,
    "n_layer": n_layer,
    "block_size": block_size,

    "learning_rate": learning_rate,
    "norm_clip": norm_clip,
    "max_iters": max_iters,
    "warmup_iters": warmup_iters,
    "min_lr": min_lr,

    "total_params": total_params,
    "test_loss": metrics["Loss"],
    "test_perplexity": metrics["Perplexity"],
}

torch.save(checkpoint, save_path)

print(f"checkpoint 已保存到: {save_path}")
print(f"模型参数数量: {total_params:,}")

checkpoint 已保存到: model/gpt_wikitext103_layer10/checkpoint.pt
模型参数数量: 64,309,504
